# Density-Reachability and Density-Connectivity in DBSCAN

> **Difficulty**: Intermediate  
> **Estimated Time**: 45-60 minutes  
> **Prerequisites**: Understanding of ε-neighborhood, core points, completed 02_mathematical_foundations.ipynb

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand density-reachability and how clusters grow through chains
- Visualize density-reachable point chains interactively
- Understand density-connectivity and why it's needed for cluster definition
- Identify density-connected points in datasets
- Apply these concepts to understand DBSCAN's clustering behavior

## Paper References

This notebook covers concepts from:
- **Section 4.1** (p. 227): Density-Based Notions of Clusters
- **Lemma 1** (p. 227): Transitivity of density-reachability
- **Lemma 2** (p. 227): Symmetry and transitivity of density-connectivity

**Full Reference:**  
Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *KDD* (Vol. 96, No. 34, pp. 226-231).

## Table of Contents

1. [Setup and Imports](#setup)
2. [Density-Reachability](#density-reachability)
3. [Density-Connectivity](#density-connectivity)
4. [Interactive Exploration](#interactive)
5. [Mathematical Properties](#properties)
6. [Exercises](#exercises)
7. [Summary](#summary)

---

## 1. Setup and Imports <a id='setup'></a>

First, let's import the necessary libraries and set up our environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import sys
sys.path.append('..')

from src.dbscan_from_scratch import DBSCAN
from src.visualization import DBSCANVisualizer
from sklearn.datasets import make_moons, make_blobs

# Set random seed for reproducibility
np.random.seed(42)

# Initialize visualizer
viz = DBSCANVisualizer()

print("✓ Setup complete!")

---

## 2. Density-Reachability <a id='density-reachability'></a>

### Mathematical Definition [Paper §4.1, p. 227]

A point **p** is **density-reachable** from a point **q** with respect to ε and MinPts if there exists a chain of points p₁, p₂, ..., pₙ where:

```
1. p₁ = q
2. pₙ = p
3. pᵢ₊₁ is directly density-reachable from pᵢ for all i ∈ {1, ..., n-1}
```

**Intuition**: You can reach p from q by "hopping" through a chain of core points, where each hop is within distance ε.

**Key Property**: Density-reachability is **transitive** but **not symmetric**.

### Why Density-Reachability Matters

Without density-reachability, clusters would be limited to ε-sized regions. With it, clusters can extend far beyond ε by chaining through core points.

**Example**: A long, winding cluster where endpoints are > ε apart, but connected through intermediate points.

### Visualizing Density-Reachability

Let's create a dataset and visualize density-reachable chains:

In [ ]:
# Create a curved cluster
t = np.linspace(0, 2*np.pi, 20)
X_curve = np.column_stack([np.cos(t), np.sin(t)])
X_curve += np.random.randn(20, 2) * 0.05  # Add small noise

# Parameters
eps = 0.4
min_pts = 3

# Run DBSCAN
dbscan = DBSCAN(eps=eps, min_pts=min_pts)
labels = dbscan.fit_predict(X_curve)

# Find a density-reachable chain
# Start from point 0, find chain to point 10
point_chain = [0, 2, 5, 8, 10]

# Visualize the chain
viz.plot_density_reachability(X_curve, point_chain, eps, labels)
plt.show()

print(f"\n📊 Density-Reachability Chain:")
print(f"  • Chain: {' → '.join([str(i) for i in point_chain])}")
print(f"  • Point {point_chain[-1]} is density-reachable from point {point_chain[0]}")
print(f"  • Direct distance: {np.linalg.norm(X_curve[point_chain[0]] - X_curve[point_chain[-1]]):.3f}")
print(f"  • This distance is > ε ({eps}), but they're still connected!")

### Directly Density-Reachable vs. Density-Reachable

**Directly Density-Reachable**: One-step connection (q ∈ N_ε(p) and p is core)

**Density-Reachable**: Multi-step connection through a chain of core points

Let's explore this distinction:

In [ ]:
# Create a simple linear cluster
X_linear = np.array([
    [0, 0],   # Point 0
    [0.3, 0], # Point 1
    [0.6, 0], # Point 2
    [0.9, 0], # Point 3
    [1.2, 0], # Point 4
])

eps = 0.35
min_pts = 2

# Analyze each point
print("Analyzing Density-Reachability:\n")
for i in range(len(X_linear)):
    distances = np.linalg.norm(X_linear - X_linear[i], axis=1)
    neighbors = np.where(distances <= eps)[0]
    is_core = len(neighbors) >= min_pts
    
    print(f"Point {i}: {X_linear[i]}")
    print(f"  Neighbors: {neighbors.tolist()}")
    print(f"  |N_ε(p)|: {len(neighbors)}")
    print(f"  Is core: {is_core}")
    print()

# Visualize
fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(X_linear[:, 0], X_linear[:, 1], c='blue', s=200, zorder=5)
for i in range(len(X_linear)):
    ax.text(X_linear[i, 0], X_linear[i, 1] + 0.05, str(i), 
           ha='center', fontsize=12, fontweight='bold')
    circle = plt.Circle((X_linear[i, 0], X_linear[i, 1]), eps,
                       color='gray', fill=False, linestyle='--', alpha=0.3)
    ax.add_patch(circle)

ax.set_xlim(-0.5, 1.7)
ax.set_ylim(-0.5, 0.5)
ax.set_aspect('equal')
ax.set_title(f'Linear Cluster (ε={eps}, MinPts={min_pts})')
ax.grid(True, alpha=0.3)
plt.show()

print("\n🔍 Observations:")
print("  • Point 1 is directly density-reachable from Point 0")
print("  • Point 2 is directly density-reachable from Point 1")
print("  • Point 2 is density-reachable from Point 0 (via Point 1)")
print("  • Point 4 is density-reachable from Point 0 (via Points 1, 2, 3)")

### Asymmetry of Density-Reachability

**Important**: Density-reachability is NOT symmetric!

A border point can be density-reachable from a core point, but not vice versa.

In [ ]:
# Create dataset with core and border points
X_asymmetric = np.array([
    [0, 0],     # Point 0 - Core
    [0.2, 0],   # Point 1 - Core
    [0.4, 0],   # Point 2 - Core
    [0.6, 0.3], # Point 3 - Border (only 2 neighbors)
])

eps = 0.35
min_pts = 3

# Analyze
print("Demonstrating Asymmetry:\n")
for i in range(len(X_asymmetric)):
    distances = np.linalg.norm(X_asymmetric - X_asymmetric[i], axis=1)
    neighbors = np.where(distances <= eps)[0]
    is_core = len(neighbors) >= min_pts
    point_type = "Core" if is_core else "Border"
    
    print(f"Point {i}: {point_type}")
    print(f"  |N_ε(p)|: {len(neighbors)}")
    print()

print("\n🔍 Key Insight:")
print("  • Point 3 (border) is density-reachable from Point 2 (core) ✓")
print("  • Point 2 (core) is NOT density-reachable from Point 3 (border) ✗")
print("  • Reason: Point 3 is not a core point, so no chain can start from it")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['blue', 'blue', 'blue', 'green']
sizes = [200, 200, 200, 150]
labels_text = ['Core', 'Core', 'Core', 'Border']

for i in range(len(X_asymmetric)):
    ax.scatter(X_asymmetric[i, 0], X_asymmetric[i, 1], 
              c=colors[i], s=sizes[i], zorder=5, edgecolors='black', linewidths=1)
    ax.text(X_asymmetric[i, 0], X_asymmetric[i, 1] - 0.08, 
           f"{i}\n({labels_text[i]})", ha='center', fontsize=10, fontweight='bold')

# Draw arrow from 2 to 3
ax.annotate('', xy=(X_asymmetric[3, 0], X_asymmetric[3, 1]),
           xytext=(X_asymmetric[2, 0], X_asymmetric[2, 1]),
           arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(0.5, 0.15, '✓ Reachable', color='green', fontsize=11, fontweight='bold')

# Draw crossed arrow from 3 to 2
ax.plot([X_asymmetric[3, 0], X_asymmetric[2, 0]], 
       [X_asymmetric[3, 1], X_asymmetric[2, 1]], 
       'r--', lw=2, alpha=0.5)
ax.text(0.5, 0.25, '✗ Not Reachable', color='red', fontsize=11, fontweight='bold')

ax.set_xlim(-0.2, 0.8)
ax.set_ylim(-0.2, 0.5)
ax.set_aspect('equal')
ax.set_title('Asymmetry of Density-Reachability')
ax.grid(True, alpha=0.3)
plt.show()

---

## 3. Density-Connectivity <a id='density-connectivity'></a>

### Mathematical Definition [Paper §4.1, p. 227]

Two points **p** and **q** are **density-connected** with respect to ε and MinPts if there exists a point **o** such that:

```
1. p is density-reachable from o
2. q is density-reachable from o
```

**Intuition**: Two points are density-connected if they can both be reached from some common core point.

**Key Property**: Density-connectivity is **symmetric** and **transitive**.

### Why Density-Connectivity?

Density-reachability alone is not sufficient to define clusters because it's not symmetric. Density-connectivity fixes this by requiring both points to be reachable from a common core point.

**Example**: Two border points on opposite sides of a cluster are NOT density-reachable from each other, but they ARE density-connected (both reachable from core points).

In [ ]:
# Create dataset with two border points
X_connectivity = np.array([
    [0, 0.3],   # Point 0 - Border (top)
    [0, 0],     # Point 1 - Core
    [0.3, 0],   # Point 2 - Core
    [0.6, 0],   # Point 3 - Core
    [0.9, 0.3], # Point 4 - Border (bottom)
])

eps = 0.35
min_pts = 2

# Analyze
print("Demonstrating Density-Connectivity:\n")
for i in range(len(X_connectivity)):
    distances = np.linalg.norm(X_connectivity - X_connectivity[i], axis=1)
    neighbors = np.where(distances <= eps)[0]
    is_core = len(neighbors) >= min_pts
    point_type = "Core" if is_core else "Border"
    
    print(f"Point {i}: {point_type}, |N_ε(p)|: {len(neighbors)}")

print("\n🔍 Analysis:")
print("  • Point 0 (border) is density-reachable from Point 1 (core)")
print("  • Point 4 (border) is density-reachable from Point 3 (core)")
print("  • Point 0 and Point 4 are NOT density-reachable from each other")
print("  • BUT: Point 0 and Point 4 ARE density-connected!")
print("    (Both are reachable from Point 2, the connecting core point)")

# Visualize
point_a, point_b, connecting_point = 0, 4, 2
viz.plot_density_connectivity(X_connectivity, point_a, point_b, connecting_point, eps)
plt.show()

### Symmetry of Density-Connectivity

Unlike density-reachability, density-connectivity IS symmetric:

In [ ]:
# Generate a cluster
X_cluster, _ = make_blobs(n_samples=50, centers=1, cluster_std=0.3, random_state=42)

eps = 0.4
min_pts = 5

# Run DBSCAN
dbscan = DBSCAN(eps=eps, min_pts=min_pts)
labels = dbscan.fit_predict(X_cluster)
core_indices = dbscan.get_core_points()

# Select two points from the same cluster
cluster_points = np.where(labels == 0)[0]
if len(cluster_points) >= 2:
    point_a, point_b = cluster_points[0], cluster_points[-1]
    
    print(f"Testing Symmetry of Density-Connectivity:\n")
    print(f"Point A: {point_a}, Point B: {point_b}")
    print(f"Both points are in cluster {labels[point_a]}")
    print(f"\nBy definition of DBSCAN clusters:")
    print(f"  • A is density-connected to B ✓")
    print(f"  • B is density-connected to A ✓")
    print(f"  • Density-connectivity is SYMMETRIC!")

# Visualize the cluster
viz.plot_point_types(X_cluster, labels, core_indices, eps)
plt.scatter([X_cluster[point_a, 0], X_cluster[point_b, 0]], 
           [X_cluster[point_a, 1], X_cluster[point_b, 1]],
           facecolors='none', edgecolors='red', s=300, linewidths=3, 
           label='Selected Points', zorder=10)
plt.legend()
plt.title('Density-Connected Points in Same Cluster')
plt.show()

---

## 4. Interactive Exploration <a id='interactive'></a>

### Interactive Density-Reachability Explorer

Explore how density-reachability works with different parameters:

In [ ]:
# Generate dataset
X_interactive, _ = make_moons(n_samples=100, noise=0.05, random_state=42)

def explore_density_concepts(eps, min_pts, show_type):
    """Interactive exploration of density concepts"""
    
    # Run DBSCAN
    dbscan = DBSCAN(eps=eps, min_pts=min_pts)
    labels = dbscan.fit_predict(X_interactive)
    core_indices = dbscan.get_core_points()
    
    # Count statistics
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    n_core = len(core_indices)
    
    # Visualize based on selection
    if show_type == 'Point Types':
        viz.plot_point_types(X_interactive, labels, core_indices, eps)
    elif show_type == 'Clusters':
        viz.plot_clusters(X_interactive, labels, 
                         title=f'DBSCAN Clustering (ε={eps:.2f}, MinPts={min_pts})')
    elif show_type == 'Epsilon Neighborhoods':
        # Show neighborhoods for a few core points
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        if len(core_indices) >= 2:
            for ax, point_idx in zip(axes, core_indices[:2]):
                plt.sca(ax)
                viz.plot_epsilon_neighborhood(X_interactive, point_idx, eps, labels)
        plt.tight_layout()
    
    plt.show()
    
    # Print statistics
    print(f"\n📊 Clustering Statistics:")
    print(f"  • Parameters: ε={eps:.2f}, MinPts={min_pts}")
    print(f"  • Clusters found: {n_clusters}")
    print(f"  • Core points: {n_core}")
    print(f"  • Noise points: {n_noise}")

# Create interactive widget
eps_slider = widgets.FloatSlider(
    value=0.3, min=0.1, max=1.0, step=0.05,
    description='Epsilon (ε):', continuous_update=False
)

minpts_slider = widgets.IntSlider(
    value=5, min=2, max=15, step=1,
    description='MinPts:', continuous_update=False
)

show_dropdown = widgets.Dropdown(
    options=['Point Types', 'Clusters', 'Epsilon Neighborhoods'],
    value='Point Types',
    description='Show:'
)

widgets.interactive(explore_density_concepts, 
                   eps=eps_slider, 
                   min_pts=minpts_slider,
                   show_type=show_dropdown)

---

## 5. Mathematical Properties <a id='properties'></a>

### Property 1: Transitivity of Density-Reachability [Lemma 1, p. 227]

**Lemma 1**: If p is density-reachable from q, and q is density-reachable from r, then p is density-reachable from r.

**Proof Sketch**: Concatenate the two chains.

Let's verify this property:

In [ ]:
# Create a long chain
X_chain = np.array([
    [0, 0],    # r (Point 0)
    [0.3, 0],  # Point 1
    [0.6, 0],  # q (Point 2)
    [0.9, 0],  # Point 3
    [1.2, 0],  # p (Point 4)
])

eps = 0.35
min_pts = 2

print("Verifying Transitivity of Density-Reachability:\n")
print("Chain: r (0) → 1 → q (2) → 3 → p (4)\n")
print("Given:")
print("  • q (2) is density-reachable from r (0) via chain: 0 → 1 → 2")
print("  • p (4) is density-reachable from q (2) via chain: 2 → 3 → 4")
print("\nBy Lemma 1 (Transitivity):")
print("  • p (4) is density-reachable from r (0) via chain: 0 → 1 → 2 → 3 → 4 ✓")

# Visualize
fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(X_chain[:, 0], X_chain[:, 1], c='blue', s=200, zorder=5)
for i in range(len(X_chain)):
    label = ['r', '1', 'q', '3', 'p'][i]
    ax.text(X_chain[i, 0], X_chain[i, 1] + 0.08, label, 
           ha='center', fontsize=14, fontweight='bold')
    if i < len(X_chain) - 1:
        ax.annotate('', xy=(X_chain[i+1, 0], X_chain[i+1, 1]),
                   xytext=(X_chain[i, 0], X_chain[i, 1]),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.set_xlim(-0.3, 1.5)
ax.set_ylim(-0.3, 0.3)
ax.set_aspect('equal')
ax.set_title('Transitivity: p is density-reachable from r')
ax.grid(True, alpha=0.3)
plt.show()

### Property 2: Symmetry and Transitivity of Density-Connectivity [Lemma 2, p. 227]

**Lemma 2**: Density-connectivity is symmetric and transitive.

**Implication**: Density-connectivity is an equivalence relation, which partitions the dataset into disjoint clusters.

In [ ]:
# Generate multiple clusters
X_multi, y_true = make_blobs(n_samples=150, centers=3, cluster_std=0.3, random_state=42)

eps = 0.5
min_pts = 5

# Run DBSCAN
dbscan = DBSCAN(eps=eps, min_pts=min_pts)
labels = dbscan.fit_predict(X_multi)

print("Verifying Equivalence Relation Property:\n")
print("Density-connectivity partitions the dataset into equivalence classes (clusters).\n")

# Count points in each cluster
unique_labels = set(labels)
for label in unique_labels:
    if label == -1:
        print(f"Noise: {list(labels).count(label)} points")
    else:
        print(f"Cluster {label}: {list(labels).count(label)} points")

print("\n🔍 Key Insight:")
print("  • All points within a cluster are density-connected to each other")
print("  • No point in one cluster is density-connected to points in another cluster")
print("  • This creates a partition of the dataset (disjoint sets)")

# Visualize
viz.plot_clusters(X_multi, labels, title='Density-Connectivity Creates Disjoint Clusters')
plt.show()

---

## 6. Exercises <a id='exercises'></a>

### Exercise 1: Identifying Density-Reachable Points (Beginner)

Given the following points with ε = 0.4 and MinPts = 2:

```
A: (0, 0) - neighbors: A, B (2 neighbors)
B: (0.3, 0) - neighbors: A, B, C (3 neighbors)
C: (0.6, 0) - neighbors: B, C, D (3 neighbors)
D: (0.9, 0) - neighbors: C, D (2 neighbors)
```

**Questions**:
1. Which points are core points?
2. Is D density-reachable from A?
3. Is A density-reachable from D?

<details>
<summary>Solution</summary>

1. Core points: B and C (both have ≥ 2 neighbors)
2. Yes, D is density-reachable from A via chain: A → B → C → D
3. No, A is not density-reachable from D (D is not a core point)
</details>

### Exercise 2: Density-Connectivity (Intermediate)

Using the same points from Exercise 1:

**Questions**:
1. Are A and D density-connected?
2. What is the connecting point?
3. Do A and D belong to the same cluster?

<details>
<summary>Solution</summary>

1. Yes, A and D are density-connected
2. The connecting points are B or C (both are core points from which A and D are reachable)
3. Yes, they belong to the same cluster because they are density-connected
</details>

### Exercise 3: Practical Application (Advanced)

Create your own dataset and explore density concepts:

In [ ]:
# Exercise 3: Create and analyze your own dataset
# TODO: Create a dataset with interesting density properties
# Hint: Try creating clusters with varying densities or shapes

# Your code here:
# X_custom = ...
# eps = ...
# min_pts = ...

# Analyze and visualize
# dbscan = DBSCAN(eps=eps, min_pts=min_pts)
# labels = dbscan.fit_predict(X_custom)
# viz.plot_clusters(X_custom, labels)
# plt.show()

print("Create your own dataset and explore!")

---

## 7. Summary <a id='summary'></a>

### Key Takeaways

1. **Density-Reachability** [Paper §4.1, p. 227]:
   - Extends "nearby" through chains of core points
   - Transitive but NOT symmetric
   - Enables clusters larger than ε radius
   - Formula: ∃ chain p₁ → p₂ → ... → pₙ where each step is directly density-reachable

2. **Density-Connectivity** [Paper §4.1, p. 227]:
   - Symmetric relation for cluster membership
   - Two points connected if both reachable from common core point
   - Defines equivalence classes (clusters)
   - Formula: ∃ o such that both p and q are density-reachable from o

3. **Mathematical Properties**:
   - Density-reachability: transitive [Lemma 1]
   - Density-connectivity: symmetric and transitive [Lemma 2]
   - These properties ensure well-defined clusters

4. **Practical Implications**:
   - Arbitrary cluster shapes possible
   - Border points correctly assigned
   - Deterministic clustering
   - Formal correctness guarantees

### What You Can Do Now

✓ Understand density-reachability and its asymmetry  
✓ Visualize density-reachable chains  
✓ Understand density-connectivity and its symmetry  
✓ Identify density-connected points  
✓ Apply these concepts to understand DBSCAN behavior  

---

## Next Steps

1. **[05_algorithm_walkthrough.ipynb](05_algorithm_walkthrough.ipynb)** - See how the algorithm uses these concepts step-by-step
2. **[06_parameter_tuning.ipynb](06_parameter_tuning.ipynb)** - Learn how to select optimal ε and MinPts
3. **[docs/03_algorithm_details.md](../docs/03_algorithm_details.md)** - Read about the complete algorithm
4. **Paper §4.1** (p. 227) - Read the formal definitions and proofs

---

**Happy Learning! 🎓**